# TTM–ETTh1 최소 재현

기본값은 **축소 smoke**입니다. 데이터 준비 → Zero-Shot → 0.5% FT → 동일한 시험 샘플 → 개선율 출력의 기능 확인용입니다. smoke는 2 updates, validation 4윈도, test 8윈도로 줄이므로 논문 결과와 직접 비교할 수 없습니다. 전체 논문은 270조건이고 본 notebook의 paper-pair도 2조건뿐입니다.

Colab GPU/Linux/Python 3.12가 필요합니다. 기존 공식 모델 adapter·학습·평가 코드를 재사용합니다. 현재 환경에서 새 GPU 실행 검증은 하지 않았습니다. 기록된 패키지/torch/CUDA를 제공하지 않는 Colab 기본 runtime은 원 실행 gate를 통과하지 않을 수 있습니다. 토큰·개인 Drive·절대경로는 사용하지 않습니다.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY = "https://github.com/Han-Youseung/tsfm-zero-few-shot-crossover.git"
SOURCE_REF = "0ce9a35539e811f1c90905c4cf11ca537d56b047"  # Tested public code revision.
repo = Path("tsfm-zero-few-shot-crossover")
if not repo.exists():
    subprocess.run(["git", "clone", REPOSITORY, str(repo)], check=True)
    subprocess.run(
        [
            "git",
            "checkout",
            "--detach",
            (SOURCE_REF if len(SOURCE_REF) == 40 else "origin/" + SOURCE_REF),
        ],
        cwd=repo,
        check=True,
    )
else:
    expected = subprocess.check_output(
        ["git", "rev-parse", (SOURCE_REF if len(SOURCE_REF) == 40 else "origin/" + SOURCE_REF)],
        cwd=repo,
        text=True,
    ).strip()
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain", "--untracked-files=no"], cwd=repo, text=True
    ).strip()
    if actual != expected or dirty:
        raise RuntimeError("Existing checkout differs; preserve it and use a fresh runtime")
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip())

In [ ]:
import uuid

SMOKE_MODE = True
# False uses full paper settings for the two conditions, not all 270.

output = Path("outputs") / (("smoke-" if SMOKE_MODE else "paper-pair-") + uuid.uuid4().hex[:8])
print(
    "SMOKE: not comparable to paper"
    if SMOKE_MODE
    else "Paper settings: only ZS and 0.5%, seed 1729"
)

In [ ]:
if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Use a Python 3.12 runtime matching the recorded model environment")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements/ttm-main.txt"],
    cwd=repo,
    check=True,
)

설치 후 runtime 재시작이 필요하다고 Colab이 알리면 재시작하고 clone/설정 셀부터 재실행하세요. GPU 가중치 다운로드는 다음 실행에서 시작됩니다. 설치 성공은 GPU 검증 성공이 아닙니다.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Select a CUDA GPU runtime; no CPU result will be reported as GPU validation"
    )
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))
subprocess.run(
    [sys.executable, "scripts/prepare_study_data.py", "--dataset", "ETTh1"], cwd=repo, check=True
)

In [ ]:
runner = "scripts/reproduce_minimal.py" if SMOKE_MODE else "scripts/reproduce_paper.py"
command = [sys.executable, runner, "--output", str(output)]
if not SMOKE_MODE:
    command += ["--scope", "pair"]
subprocess.run(command, cwd=repo, check=True)  # CPU plan

다음 셀은 **명시적 모델 실행**입니다. 기존 결과를 덮어쓰지 않고 새 output을 사용합니다. 테스트 값으로 설정을 고르지 마세요. paper-pair는 원 runtime gate를 그대로 적용합니다.

In [ ]:
subprocess.run(command + ["--execute"], cwd=repo, check=True)

In [ ]:
import json

if SMOKE_MODE:
    result = json.loads((repo / output / "smoke_result.json").read_text(encoding="utf-8"))
    print(
        {
            k: result[k]
            for k in [
                "status",
                "paper_comparable",
                "zero_nmae",
                "ft_nmae",
                "improvement_percent",
                "actual_steps",
                "best_step",
                "test_windows",
            ]
        }
    )
else:
    rows = [
        json.loads(
            (repo / output / ("ttm-ETTh1-h96-s1729-r" + rate) / "result.json").read_text(
                encoding="utf-8"
            )
        )
        for rate in ["0", "0.005"]
    ]
    assert rows[0]["test"]["window_hash"] == rows[1]["test"]["window_hash"]
    zero, few = [r["test"]["metrics"]["macro"]["normalized_mae"] for r in rows]
    print({"zero_nmae": zero, "ft_nmae": few, "improvement_percent": 100 * (zero - few) / zero})

결과는 수집 비용 감소를 측정하지 않습니다. 0.5%는 전체 train 기간에서 선택한 윈도 비율입니다. 100% 풀도 전체 순회·수렴을 뜻하지 않습니다. 모델 개선 폭과 최종 오차를 구분하세요. checkpoint는 로컬 런타임에 남으며 자동으로 업로드하지 않습니다.